# Lab 5 — Estimation: G-Computation, IPW, AIPW, and DML

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sreent/causal-ai-for-smart-manufacturing/blob/main/labs/ch05/lab05.ipynb)

**Companion lab to Chapter 5.** When back-door identification holds, what is the best estimator? This lab implements four estimators from scratch on the same data — *G-computation* (outcome-model-based), *IPW* (propensity-score-based), *AIPW* (doubly-robust), and *DML* (cross-fitted doubly-robust) — and shows how each handles complex outcome surfaces, imbalanced treatment, and positivity violations.

The chapter (§5.9) lays out a smart-manufacturing line example: a controllable parameter $X$ (conveyor speed) affects first-pass yield $Y$, with observed confounders $Z$ (product mix, ambient temperature, operator skill, day of week, incoming defect rate). The team estimates the ATE four ways and reports a converging $\sim +2\%$ effect. This lab builds that example end-to-end.

## What you'll do

1. **Build the SCM** for the smart-manufacturing line with five observed confounders.
2. **Estimate the ATE the production way**: a few lines of EconML's `LinearDML`. This is the workflow you would run on real data; everything that follows is the "under the hood" view.
3. **Implement the four estimators from scratch**: *G-computation* (outcome-model-based), *IPW* (propensity-score-based), *AIPW* (doubly-robust), and *DML* (cross-fitted doubly-robust). These mirror what EconML does internally and let you see where the gains come from.
4. **Diagnose overlap**: plot the propensity-score histogram, identify the overlap region.
5. **Stress-test with a positivity violation**: restrict the data to certain (treatment, covariate) combinations and watch IPW go badly wrong; DR/DML rescue it.
6. **Compare**: EconML's number, the four manual numbers, and the oracle truth should all line up when assumptions hold.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (8, 5)

## Part 1 — The smart-manufacturing SCM

A line that produces a single product runs multiple shifts per day. The line lead sets the conveyor speed at the start of each shift, choosing between "high" ($X=1$) and "low" ($X=0$) based on the day's product mix, ambient conditions, and incoming material. First-pass yield is recorded at end of shift.

Confounders we observe:

- $Z_1$ — product-mix code (continuous in this synthetic version; represents complexity of the current run)
- $Z_2$ — ambient temperature deviation from setpoint (°C)
- $Z_3$ — operator skill (1 = senior, 0 = junior)
- $Z_4$ — day-of-week effect (encoded as Mon–Sun cyclical)
- $Z_5$ — incoming lot's defect rate

The line lead's choice rule (the *behavior policy*): high speed is more likely when product mix is simple, ambient is at setpoint, the operator is senior, and the incoming lot looks clean.

The structural equation for yield: $Y$ depends on $X$ and on the same $Z_1, \dots, Z_5$. The *causal* effect of $X$ on $Y$ is small but positive — about $+0.02$ on the FPY fraction — high speed gives slightly higher yield *holding everything else constant*. In the observational data the confounders amplify this signal: high speed is picked when conditions are favorable, and favorable conditions independently raise yield, so the *observational* difference between treated and control is roughly $+0.09$, four times the truth. Confounding doesn't always flip the sign; here it inflates the magnitude. The four identified estimators should each recover $+0.02$.

In [ ]:
def gen_data(n, rng, positivity_restrict=False):
    """Generate observational data from the line SCM.

    If positivity_restrict=True, force X = 0 for senior operators on simple-mix shifts
    (a positivity violation: certain (X, Z) combinations never appear).
    """
    Z1 = rng.normal(0, 1, n)                     # product mix (high = complex)
    Z2 = rng.normal(0, 1, n)                     # ambient temp deviation
    Z3 = rng.binomial(1, 0.5, n)                 # operator skill (senior=1)
    Z4 = rng.uniform(0, 2*np.pi, n)              # day-of-week phase
    Z5 = np.abs(rng.normal(0, 1, n)) * 0.1       # incoming defect rate

    # Line lead's high-speed choice (behavior policy)
    logit_x = (-0.5 * Z1 - 0.3 * Z2 + 0.6 * Z3 + 0.2 * np.cos(Z4) - 5 * Z5)
    prob_x = 1 / (1 + np.exp(-logit_x))
    X = rng.binomial(1, prob_x)

    # Positivity restriction: never run high speed on simple-mix senior shifts
    if positivity_restrict:
        force_low = (Z1 < -0.5) & (Z3 == 1)
        X = np.where(force_low, 0, X)

    # True ATE = +0.02 (high speed adds 2 percentage points to FPY)
    # Yield depends on X and the confounders
    y_logit = (0.5 + 0.1 * X
               - 0.3 * Z1 - 0.2 * Z2 + 0.4 * Z3 + 0.1 * np.cos(Z4) - 4 * Z5
               + rng.normal(0, 0.2, n))
    Y = 1 / (1 + np.exp(-y_logit))           # FPY as a fraction in [0, 1]
    return pd.DataFrame({"X": X, "Y": Y,
                         "Z1": Z1, "Z2": Z2, "Z3": Z3, "Z4": Z4, "Z5": Z5,
                         "prob_x_true": prob_x})

n = 5000
data = gen_data(n, rng)
print(f"Treated fraction:    {data['X'].mean():.3f}")
print(f"Yield range:         [{data['Y'].min():.3f}, {data['Y'].max():.3f}]")
print()
print(f"Naive E[Y|X=1] - E[Y|X=0] = {data[data['X']==1]['Y'].mean() - data[data['X']==0]['Y'].mean():+.4f}")
print(f"(This is the *observational* difference, contaminated by confounding.)")

## Part 2 — The true ATE

Before estimating, compute the truth via Monte Carlo on the SCM.

In [ ]:
def true_ate(rng, n=50_000):
    Z1 = rng.normal(0, 1, n); Z2 = rng.normal(0, 1, n)
    Z3 = rng.binomial(1, 0.5, n); Z4 = rng.uniform(0, 2*np.pi, n)
    Z5 = np.abs(rng.normal(0, 1, n)) * 0.1
    # Y under do(X=1)
    yl1 = 0.5 + 0.1*1 - 0.3*Z1 - 0.2*Z2 + 0.4*Z3 + 0.1*np.cos(Z4) - 4*Z5 + rng.normal(0, 0.2, n)
    yl0 = 0.5 + 0.1*0 - 0.3*Z1 - 0.2*Z2 + 0.4*Z3 + 0.1*np.cos(Z4) - 4*Z5 + rng.normal(0, 0.2, n)
    return (1/(1+np.exp(-yl1)) - 1/(1+np.exp(-yl0))).mean()

ATE_true = true_ate(np.random.default_rng(99))
print(f"True ATE (Monte Carlo from SCM): {ATE_true:+.4f}")

The true ATE is small but positive — ~2 percentage points on FPY, consistent with the chapter's §5.9 numbers. The naive observational difference (Part 1) was ~$+0.09$, four times the true value. Confounding has inflated the magnitude. A team that took the naive difference as the effect would over-promise the rollout's impact and under-deliver.

Now we estimate the true ATE from observational data alone — first the production workflow, then the four estimators that sit inside it.

## Part 3 — The production workflow: EconML `LinearDML`

In a real deployment you would not implement DML from scratch. You would call EconML's `LinearDML`, which does the cross-fitted doubly-robust estimation in a few lines, returns a point estimate plus a valid confidence interval, and lets you swap nuisance learners (gradient boosting, random forest, neural nets) without touching the estimator code.

This is the workflow. Everything in Parts 4–7 is "what's happening inside `LinearDML`" — useful to understand, but not what you write on the actual project.

In [ ]:
try:
    from econml.dml import LinearDML
    have_econml = True
except ImportError:
    have_econml = False
    print("EconML not installed. The %pip install at the top should have fixed this in Colab;")
    print("locally, run `pip install econml`. Skipping Part 3 (the rest of the lab still runs).")
from sklearn.ensemble import GradientBoostingClassifier as GBC

# Columns we treat as observed confounders, used throughout the rest of the lab.
ZCOLS = ["Z1", "Z2", "Z3", "Z4", "Z5"]

if have_econml:
    est = LinearDML(
        model_y=GradientBoostingRegressor(random_state=0, n_estimators=100),
        model_t=GBC(random_state=0, n_estimators=100),
        discrete_treatment=True,
        cv=5,
        random_state=0,
    )
    est.fit(Y=data["Y"].values, T=data["X"].values, X=None, W=data[ZCOLS].values)

    # Point estimate (LinearDML.const_marginal_ate when X is None gives the marginal ATE)
    ate_econml = float(np.atleast_1d(est.const_marginal_ate()).ravel()[0])
    # Confidence interval (EconML's inference framework)
    ate_lower, ate_upper = est.const_marginal_ate_interval(alpha=0.05)
    ate_lower = float(np.atleast_1d(ate_lower).ravel()[0])
    ate_upper = float(np.atleast_1d(ate_upper).ravel()[0])

    print(f"EconML LinearDML ATE: {ate_econml:+.4f}   95% CI: [{ate_lower:+.4f}, {ate_upper:+.4f}]")
    print(f"True ATE (oracle):    {ATE_true:+.4f}")

That is the production answer: ATE point estimate, confidence interval, both produced by a 6-line call to a maintained library. The estimator handles cross-fitting, nuisance modeling, and confidence-interval construction without you needing to write any of it.

The rest of the lab opens this box. *Why* does `LinearDML` work? What's a "doubly-robust" estimator? How does cross-fitting prevent the bias that AIPW would have without it? You don't *need* the answers to use the library — but you do need them to (a) recognize when the library's assumptions fail, (b) decide what to put in `model_y` and `model_t`, and (c) defend the result to a skeptical reviewer.

## Part 4 — Under the hood: G-Computation

Fit $\hat\mu(x, z) = E[Y \mid X = x, Z = z]$ on the data, then average the counterfactual contrasts:

$$\hat\tau_{\text{G-comp}} = \frac{1}{n}\sum_{i=1}^n \left[\hat\mu(1, Z_i) - \hat\mu(0, Z_i)\right].$$

In [ ]:
def g_computation(data, learner=None):
    if learner is None:
        learner = GradientBoostingRegressor(random_state=0, n_estimators=100)
    X_features = data[["X"] + ZCOLS].values
    mu = learner.fit(X_features, data["Y"].values)
    n = len(data)
    Z = data[ZCOLS].values
    mu1 = mu.predict(np.column_stack([np.ones(n), Z]))
    mu0 = mu.predict(np.column_stack([np.zeros(n), Z]))
    return (mu1 - mu0).mean()

ate_gcomp = g_computation(data)
print(f"G-computation ATE: {ate_gcomp:+.4f}  (true: {ATE_true:+.4f})")

G-computation recovers the truth. Its main risk: if the outcome model $\hat\mu$ is misspecified, the bias propagates directly into the estimate. With a flexible learner (gradient boosting) on this synthetic data, the outcome surface is well-captured.

## Part 5 — Under the hood: Inverse Probability Weighting

Fit the propensity score $\hat e(z) = P(X = 1 \mid Z = z)$. Then weight treated units by $1/\hat e(z)$ and control units by $1/(1 - \hat e(z))$ to balance the pseudo-population. The IPW estimator:

$$\hat\tau_{\text{IPW}} = \frac{1}{n}\sum_{i=1}^n \left[\frac{X_i Y_i}{\hat e(Z_i)} - \frac{(1-X_i) Y_i}{1 - \hat e(Z_i)}\right].$$

In [ ]:
def ipw(data, trim_threshold=0.05):
    learner = GradientBoostingClassifier(random_state=0, n_estimators=100)
    e_hat = learner.fit(data[ZCOLS].values, data["X"].values).predict_proba(data[ZCOLS].values)[:, 1]
    # Trim propensities to [trim_threshold, 1 - trim_threshold] for stability
    e_hat_trimmed = np.clip(e_hat, trim_threshold, 1 - trim_threshold)
    X, Y = data["X"].values, data["Y"].values
    w = X / e_hat_trimmed - (1 - X) / (1 - e_hat_trimmed)
    return (w * Y).mean(), e_hat

ate_ipw, e_hat = ipw(data)
print(f"IPW ATE:           {ate_ipw:+.4f}  (true: {ATE_true:+.4f})")

Inspect the propensity-score distribution. Good overlap (propensity densities for treated and control overlap across [0, 1]) is what makes IPW work. Poor overlap (separation, propensities near 0 or 1) means the weights blow up and IPW becomes high-variance.

In [ ]:
fig, ax = plt.subplots()
ax.hist(e_hat[data["X"] == 1], bins=40, alpha=0.5, label="Treated (X=1)", color="C0")
ax.hist(e_hat[data["X"] == 0], bins=40, alpha=0.5, label="Control (X=0)", color="C1")
ax.set_xlabel("Estimated propensity e_hat(Z)")
ax.set_ylabel("Count")
ax.set_title("Propensity score overlap")
ax.legend()
plt.show()

The two distributions overlap broadly — neither group is concentrated at the extremes — so positivity is well-satisfied and IPW is reliable.

## Part 6 — Under the hood: AIPW (doubly robust)

The augmented IPW estimator combines G-computation and IPW. The key formula:

$$\hat\tau_{\text{AIPW}} = \frac{1}{n}\sum_{i=1}^n \left[\hat\mu(1, Z_i) - \hat\mu(0, Z_i) + \frac{X_i (Y_i - \hat\mu(1, Z_i))}{\hat e(Z_i)} - \frac{(1-X_i)(Y_i - \hat\mu(0, Z_i))}{1 - \hat e(Z_i)}\right].$$

AIPW is *consistent* if either $\hat\mu$ or $\hat e$ is correctly specified. It usually has lower variance than IPW alone.

In [ ]:
def aipw(data):
    Y, X = data["Y"].values, data["X"].values
    Z = data[ZCOLS].values
    # Outcome model
    mu = GradientBoostingRegressor(random_state=0, n_estimators=100).fit(
        np.column_stack([X, Z]), Y)
    mu1 = mu.predict(np.column_stack([np.ones_like(X), Z]))
    mu0 = mu.predict(np.column_stack([np.zeros_like(X), Z]))
    # Propensity model
    e_hat = GradientBoostingClassifier(random_state=0, n_estimators=100).fit(
        Z, X).predict_proba(Z)[:, 1]
    e_hat = np.clip(e_hat, 0.05, 0.95)
    # AIPW formula
    correction1 = X * (Y - mu1) / e_hat
    correction0 = (1 - X) * (Y - mu0) / (1 - e_hat)
    return (mu1 - mu0 + correction1 - correction0).mean()

ate_aipw = aipw(data)
print(f"AIPW ATE:          {ate_aipw:+.4f}  (true: {ATE_true:+.4f})")

## Part 7 — Under the hood: DML (cross-fitted AIPW)

Cross-fitting prevents the nuisance models from overfitting the fold they predict on. The procedure: split the data into $K$ folds, train nuisances on $K-1$ folds, predict on the held-out fold, repeat. The AIPW formula is then applied to the out-of-fold predictions.

In [ ]:
def dml(data, n_folds=5):
    Y, X = data["Y"].values, data["X"].values
    Z = data[ZCOLS].values
    n = len(data)
    mu1_oof = np.zeros(n); mu0_oof = np.zeros(n); e_oof = np.zeros(n)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    for train_idx, test_idx in kf.split(Z):
        # Outcome model on training fold
        mu = GradientBoostingRegressor(random_state=0, n_estimators=100).fit(
            np.column_stack([X[train_idx], Z[train_idx]]), Y[train_idx])
        mu1_oof[test_idx] = mu.predict(np.column_stack([np.ones(len(test_idx)), Z[test_idx]]))
        mu0_oof[test_idx] = mu.predict(np.column_stack([np.zeros(len(test_idx)), Z[test_idx]]))
        # Propensity model on training fold
        e_oof[test_idx] = GradientBoostingClassifier(random_state=0, n_estimators=100).fit(
            Z[train_idx], X[train_idx]).predict_proba(Z[test_idx])[:, 1]

    e_oof = np.clip(e_oof, 0.05, 0.95)
    correction1 = X * (Y - mu1_oof) / e_oof
    correction0 = (1 - X) * (Y - mu0_oof) / (1 - e_oof)
    return (mu1_oof - mu0_oof + correction1 - correction0).mean()

ate_dml = dml(data)
print(f"DML ATE:           {ate_dml:+.4f}  (true: {ATE_true:+.4f})")

## Part 8 — Compare: library, manual estimators, oracle

In [ ]:
print(f"True ATE (oracle):           {ATE_true:+.5f}")
print()
print(f"Naive obs. difference:       {data[data['X']==1]['Y'].mean() - data[data['X']==0]['Y'].mean():+.5f}")
print(f"Manual G-computation:        {ate_gcomp:+.5f}")
print(f"Manual IPW (trimmed):        {ate_ipw:+.5f}")
print(f"Manual AIPW:                 {ate_aipw:+.5f}")
print(f"Manual DML (5-fold):         {ate_dml:+.5f}")
print(f"EconML LinearDML:            {ate_econml:+.5f}    95% CI: [{ate_lower:+.5f}, {ate_upper:+.5f}]")

All four identified estimators recover the truth (~$+0.022$, or roughly $+2$ percentage points on FPY — matching the chapter's §5.9 numbers). The naive observational difference is $\sim +0.09$ — four times the truth, the right sign but the wrong magnitude. A team that took the naive difference as the effect would promise the rollout would deliver $+9$ pp and watch only $+2$ land. The four identified estimators agree to within a couple of percentage-points-of-percentage-points of each other. When they agree, you have convergent evidence. When they disagree, the *pattern* of disagreement diagnoses where the assumptions or models are failing — see the chapter §5.9 for the diagnosis grid.

## Part 9 — Stress-test with positivity violation

Now generate data with a positivity violation: certain $(X, Z)$ combinations never appear. Specifically, force senior operators on simple-mix shifts to always run low speed. Watch the estimators degrade.

In [ ]:
data_bad = gen_data(n, np.random.default_rng(7), positivity_restrict=True)

# What happens to overlap?
e_hat_bad = GradientBoostingClassifier(random_state=0, n_estimators=100).fit(
    data_bad[ZCOLS].values, data_bad["X"].values).predict_proba(data_bad[ZCOLS].values)[:, 1]

fig, ax = plt.subplots()
ax.hist(e_hat_bad[data_bad["X"] == 1], bins=40, alpha=0.5, label="Treated", color="C0")
ax.hist(e_hat_bad[data_bad["X"] == 0], bins=40, alpha=0.5, label="Control", color="C1")
ax.set_xlabel("Estimated propensity e_hat(Z)")
ax.set_ylabel("Count")
ax.set_title("Propensity overlap with positivity violation")
ax.legend()
plt.show()

print(f"True ATE:        {ATE_true:+.4f}")
print(f"G-comp:          {g_computation(data_bad):+.4f}")
print(f"IPW (trimmed):   {ipw(data_bad)[0]:+.4f}")
print(f"AIPW:            {aipw(data_bad):+.4f}")
print(f"DML:             {dml(data_bad):+.4f}")

With the positivity violation, propensity scores pile up near 0 for senior-and-simple-mix shifts (they are *always* in the control group). IPW's behavior here is more dramatic than just "higher variance" — in this run IPW flips to a **negative** estimate ($\sim -0.075$, the wrong sign by a wide margin) because the trimmed weights misrepresent the few residual treated units in the constrained region. AIPW and DML, in contrast, recover the right sign and roughly the right magnitude — the doubly-robust correction uses the outcome model to fill in where positivity is poor. This is exactly the "DR rescues IPW under positivity stress" story the chapter promises.

In practice, positivity violations are diagnosed by *inspecting* the propensity histogram. If you see concentration near 0 or 1 for one group, the estimator's reliability in that region is poor. Trim, restrict the analysis to the overlap region, or — best — switch to a doubly-robust estimator that doesn't rely solely on the propensity.

## Part 10 — A second EconML estimator: `CausalForestDML`

EconML's `CausalForestDML` gives the same ATE plus *per-unit* treatment effects with confidence intervals — useful when you suspect heterogeneity (which Lab 6 explores in depth).

In [ ]:
if have_econml:
    from econml.dml import CausalForestDML

    cf = CausalForestDML(
        model_y=GradientBoostingRegressor(random_state=0, n_estimators=100),
        model_t=GBC(random_state=0, n_estimators=100),
        discrete_treatment=True, cv=3, random_state=0, n_estimators=300,
    )
    cf.fit(Y=data["Y"].values, T=data["X"].values, X=data[ZCOLS].values)
    ate_cf = float(np.atleast_1d(cf.ate(X=data[ZCOLS].values)).ravel()[0])
    ate_cf_lower, ate_cf_upper = cf.ate_interval(X=data[ZCOLS].values, alpha=0.05)
    print(f"EconML CausalForestDML ATE: {ate_cf:+.4f}   95% CI: "
          f"[{float(np.atleast_1d(ate_cf_lower).ravel()[0]):+.4f}, "
          f"{float(np.atleast_1d(ate_cf_upper).ravel()[0]):+.4f}]")
    print(f"EconML LinearDML ATE:       {ate_econml:+.4f}")
    print(f"True ATE:                   {ATE_true:+.4f}")
else:
    print("Skipping: EconML not installed.")

## Reflection

**No single estimator is always best.** G-computation requires a correct outcome model. IPW requires a correct propensity model. AIPW requires *either* (the double robustness property). DML adds cross-fitting for valid confidence intervals with ML nuisances. Pick based on which model you have more confidence in, and report all four when sample size allows.

**Convergent estimates are evidence of identification.** When the four estimators agree, the assumptions are mutually consistent with the data. When they disagree, the *direction* of disagreement diagnoses where the model is failing.

**Propensity diagnostics are mandatory.** Always plot the propensity histogram before reporting IPW or AIPW. Concentration near the extremes means positivity is marginal, weights are unstable, and the estimator is unreliable there.

## Exercises

1. **Misspecify the outcome model.** Replace the gradient booster with a *linear* outcome regression (no nonlinear interactions). Rerun G-computation, AIPW, DML. Which estimator is most affected? Which is the least?

   <details><summary>Solution</summary>

   G-computation degrades the most: its only signal is the (now-wrong) outcome model. AIPW recovers — the IPW residual correction handles the outcome-model error (DR property). DML with the same linear outcome model also recovers, and gets valid CIs even when the ML propensity is wrong. The lesson: model misspecification on one nuisance is rescued by the other. DR is the practical default.
   </details>

2. **Misspecify the propensity model.** Replace the gradient-boosting classifier with a *linear* logistic regression. Which estimator is most affected?

   <details><summary>Solution</summary>

   IPW degrades because the trimmed weights now misrepresent the true propensity. G-computation is unaffected (it doesn't use the propensity). AIPW and DML still recover via the correct outcome model. This is the symmetric DR rescue from Ch 11 §11.9 reappearing in the ATE setting.
   </details>

3. **Extreme imbalance.** Modify the SCM so 95% of shifts run low speed ($P(X=1) = 0.05$). Rerun all four estimators. Which is most affected? Why? Make a propensity histogram.

   <details><summary>Solution</summary>

   IPW variance explodes — the 5% treated units carry weights of $\sim 1/0.05 = 20$ apiece. Trimming reduces variance but introduces bias. Propensity histogram shows treated units piled near $\hat e(z) \approx 0.05$–$0.2$ with very few above. G-comp is the most stable point estimate; AIPW/DML give the best bias-variance trade-off. Any IPW-style estimator's CI widens significantly.
   </details>

4. **Real data.** Try the Bosch Production Line Performance dataset (Kaggle). Pick a single station-level parameter as the treatment, choose an adjustment set from upstream variables based on a sketched DAG, run all four estimators.

   <details><summary>Solution</summary>

   Sketch: `kaggle competitions download -c bosch-production-line-performance`. Pick a station-level signal (e.g., `L0_S22_F545`) binarized at its median as $X$; end-of-line pass/fail as $Y$; upstream station signals as $Z$. Run G-comp/IPW/AIPW/DML via EconML. The DAG cannot be fully validated (Bosch did not release station semantics), so include a sensitivity analysis (Ch 13) over plausible alternative DAGs. Expected: estimates are noisy, the four estimators disagree, and the sensitivity-bounded interval is wide enough to include "no effect."
   </details>

## What's next

Lab 6 turns from average effects (ATE) to *conditional* average effects (CATE) — the heterogeneity of treatment effects across subpopulations. The meta-learners (S-, T-, X-, R-, DR-) and causal forests build directly on the nuisance machinery you just implemented.